In [ ]:
import os

from transformers import DistilBertTokenizerFast, TrainingArguments, Trainer

from semevalpolar.llm.data_utils import read_dataset, split_dataframe
from semevalpolar.utils import get_project_root

import numpy as np
import evaluate

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

In [ ]:
df_data = read_dataset(
    os.path.join(
        get_project_root(), "data", "dev_phase", "subtask1", "train", "eng.csv"
    )
)

In [ ]:
train_df, val_df, test_df = split_dataframe(df_data)

train_df = train_df.rename(columns={"polarization": "label"})
val_df = val_df.rename(columns={"polarization": "label"})
test_df = test_df.rename(columns={"polarization": "label"})

for df in [train_df, val_df, test_df]:
    df["label"] = df["label"].astype(int)

In [ ]:
train_df.columns

In [ ]:
from datasets import DatasetDict, Dataset

dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df),
        "validation": Dataset.from_pandas(val_df),
        "test": Dataset.from_pandas(test_df),
    }
)

dataset = dataset.remove_columns("__index_level_0__")

In [ ]:
def tokenize(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

In [ ]:
dataset = dataset.map(tokenize, batched=True)

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

In [ ]:
metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # convert the logits to their predicted class
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments(
    output_dir=os.path.join(get_project_root(), "predictions", "finetuning"),
    eval_strategy="epoch",
    push_to_hub=False,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()